# 2-1-5 모델 학습과 성능 평가
교과서 **76~81쪽**

지난 시간에는 k-NN이 **가까운 이웃의 다수결**임을 봤습니다.  
오늘은 표를 **훈련/시험으로 나누고**, 실제로 학습한 뒤 **점수와 혼동행렬**을 읽습니다.

| 오늘 할 일 | 왜 필요한가? |
|---|---|
| 분할 → 학습 → 정확도·혼동행렬 → k 바꿔 보기 | 맞힌 비율만 보면 무엇이 틀렸는지 모릅니다 |

**준비물:** `Iris1.csv` · scikit-learn (`!pip install scikit-learn`)  
**실행:** 위에서부터 ▶.


## 0. 데이터와 라이브러리


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report

df = pd.read_csv('Iris1.csv', encoding='cp949')
X = df[['꽃잎 길이', '꽃잎 너비']]   # 핵심 속성
y = df['종류']
print(X.shape, y.value_counts().to_dict())


---

# 1. 훈련 데이터와 시험 데이터로 나누기 (76쪽)

보통 훈련 60~80%, 시험 20~40%입니다.  
`Iris1_데이터살펴보기`에서 본 것처럼, **섞지 않고 앞만 자르면** 시험에 한 종류만 남을 수 있습니다. `shuffle`이 기본으로 켜져 있습니다.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7, stratify=y
)
print('훈련:', X_train.shape, y_train.value_counts().to_dict())
print('시험 :', X_test.shape, y_test.value_counts().to_dict())


> **생각하기 1**
> `stratify=y`를 넣는 이유는? 지난 시간에 앞 80%만 잘랐을 때 무슨 일이 있었나요?
> →


---

# 2. 모델 만들고 학습하기

과대적합: 훈련만 너무 잘 맞춤 / 과소적합: 너무 단순함 / 적절: 시험에서도 비슷하게 맞힘


In [ ]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)
print('훈련 정확도:', round(model.score(X_train, y_train), 3))
print('시험 정확도:', round(model.score(X_test, y_test), 3))


> **생각하기 2**
> 훈련 점수가 1.0인데 시험이 많이 낮으면 과대적합/과소적합 중 어디에 가깝나요?
> →


---

# 3. 혼동행렬과 지표 (78~79쪽)

정확도만 같으면 같은 모델이 아닙니다.

- **질병**: 있는데 없다고 함(FN)이 치명 → **재현율**  
- **스팸**: 아닌데 스팸이라 함(FP)이 문제 → **정밀도**

정확도 = (TP+TN)/전체 · 정밀도 = TP/(TP+FP) · 재현율 = TP/(TP+FN)


In [ ]:
pred = model.predict(X_test)
print(confusion_matrix(y_test, pred, labels=['세토사', '버시컬러', '버지니카']))
print()
print(classification_report(y_test, pred, digits=3))


In [ ]:
# 교과서 핵심: 정확도는 같아도 중요한 지표가 다를 수 있다
질병 = pd.DataFrame([[40, 10], [20, 30]],
                    index=['실제 있음', '실제 없음'],
                    columns=['예측 있음', '예측 없음'])
print('질병 분류 (FN=20이 큼)')
display(질병)
TP, FN, FP, TN = 40, 20, 10, 30
print('정확도:', round((TP+TN)/100, 2), ' 재현율:', round(TP/(TP+FN), 2), ' 정밀도:', round(TP/(TP+FP), 2))


> **생각하기 3**
> 정확도가 0.69로 같은 스팸 모델이 있다면, 스팸에서 더 중요하게 볼 지표는 정밀도인가요 재현율인가요?
> →


---

# 4. k를 바꿔 개선하고, 새 꽃 예측하기


In [ ]:
기록 = []
for k in range(1, 20, 2):
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train, y_train)
    기록.append({'k': k, '시험': round(m.score(X_test, y_test), 3)})
print(pd.DataFrame(기록).to_string(index=False))

print('새 꽃 [4.9, 1.6] →', model.predict([[4.9, 1.6]])[0])


> **생각하기 4**
> 시험 점수가 가장 높은 k는? k=1과 가장 큰 k의 차이는 무엇 때문이라고 생각하나요?
> →


---

# 오늘 정리

- [ ] 훈련/시험으로 나누는 이유를 말할 수 있다.
- [ ] 정확도만으로 부족할 때 정밀도·재현율을 고를 수 있다.
- [ ] k를 바꿔 점수를 비교할 수 있다.

다음 시간(`2-1-6`)에는 **숫자를 예측하는 선형 회귀**와 **군집**을 합니다.
